In [1]:
# ============================================================
# 07_final_model.ipynb
# Frozen W5 Extra Trees — Final Development Model
# ============================================================

from pathlib import Path
import json
import time

import numpy as np
import pandas as pd

from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

PROJECT_ROOT = Path("..")

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
SPLIT_DIR = DATA_DIR / "splits"
MODEL_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT.resolve())
print("Processed data:", PROCESSED_DIR.resolve())
print("Model directory:", MODEL_DIR.resolve())
print("Results directory:", RESULTS_DIR.resolve())

Project root: /Users/akashray/Lithology_Predictor
Processed data: /Users/akashray/Lithology_Predictor/data/processed
Model directory: /Users/akashray/Lithology_Predictor/models
Results directory: /Users/akashray/Lithology_Predictor/results


In [2]:
# ============================================================
# Cell 2 — Load data and verify official split
# ============================================================

# Load engineered features
df = pd.read_parquet(
    PROCESSED_DIR / "engineered_features.parquet"
)

# Load official FORCE 2020 split manifest
split_manifest = pd.read_csv(
    SPLIT_DIR / "force2020_split.csv"
)

# Load training-only preprocessing parameters
with open(
    PROCESSED_DIR / "preprocessing_params.json",
    "r"
) as f:
    preprocessing_params = json.load(f)

print("Engineered dataset shape:", df.shape)
print("Number of wells:", df["WELL"].nunique())

print("\nOfficial split:")
print(split_manifest["SPLIT"].value_counts())

print("\nWells per split:")
print(
    split_manifest.groupby("SPLIT")["WELL"]
    .nunique()
    .sort_index()
)

print("\nSplit manifest preview:")
display(split_manifest.head())

print("\nPreprocessing parameters:")
print(json.dumps(preprocessing_params, indent=2))

Engineered dataset shape: (1430974, 46)
Number of wells: 118

Official split:
SPLIT
TRAIN         98
PUBLIC        10
BLIND_TEST    10
Name: count, dtype: int64

Wells per split:
SPLIT
BLIND_TEST    10
PUBLIC        10
TRAIN         98
Name: WELL, dtype: int64

Split manifest preview:


,WELL,SPLIT
0,15_9-13,TRAIN
1,15_9-14,PUBLIC
2,15_9-15,TRAIN
3,15_9-17,TRAIN
4,15_9-23,BLIND_TEST



Preprocessing parameters:
{
  "model_features": [
    "GR",
    "RDEP_LOG10",
    "RMED_LOG10",
    "DTC",
    "RHOB"
  ],
  "train_medians": {
    "GR": 69.434272766,
    "RDEP_LOG10": 0.17010088339122542,
    "RMED_LOG10": 0.16556766517280075,
    "DTC": 109.1400795,
    "RHOB": 2.3334783316000003
  },
  "log_transformed": [
    "RDEP",
    "RMED"
  ],
  "removed_if_all_features_missing": true,
  "invalid_rules": {
    "GR": "< 0 -> NaN",
    "DTC": "<= 0 -> NaN",
    "RHOB": "<= 0 -> NaN",
    "RDEP": "<= 0 -> NaN",
    "RMED": "<= 0 -> NaN"
  }
}


In [5]:
# Check whether SPLIT already exists and inspect the merge result

print("SPLIT in engineered dataset:", "SPLIT" in df.columns)
print("SPLIT in manifest:", "SPLIT" in split_manifest.columns)

print("\nColumns containing 'SPLIT':")
print([c for c in df.columns if "SPLIT" in c.upper()])

# Check whether the manifest has unique wells
print("\nUnique wells in manifest:", split_manifest["WELL"].nunique())
print("Rows in manifest:", len(split_manifest))
print("Duplicate wells:", split_manifest["WELL"].duplicated().sum())

SPLIT in engineered dataset: True
SPLIT in manifest: True

Columns containing 'SPLIT':
['SPLIT']

Unique wells in manifest: 118
Rows in manifest: 118
Duplicate wells: 0


In [8]:
# ============================================================
# Diagnose mismatch between dataset SPLIT and official manifest
# ============================================================

manifest_map = (
    split_manifest[["WELL", "SPLIT"]]
    .set_index("WELL")["SPLIT"]
)

dataset_map = (
    df[["WELL", "SPLIT"]]
    .drop_duplicates()
    .set_index("WELL")["SPLIT"]
)

comparison = pd.DataFrame({
    "dataset_split": dataset_map,
    "official_split": manifest_map
})

mismatches = comparison[
    comparison["dataset_split"] != comparison["official_split"]
].sort_index()

print("Number of mismatched wells:", len(mismatches))

print("\nMismatched wells:")
display(mismatches)

print("\nDataset SPLIT counts:")
print(dataset_map.value_counts())

print("\nOfficial manifest counts:")
print(manifest_map.value_counts())

Number of mismatched wells: 118

Mismatched wells:


,dataset_split,official_split
WELL,,
15_9-13,train,TRAIN
15_9-14,train,PUBLIC
15_9-15,train,TRAIN
15_9-17,train,TRAIN
15_9-23,train,BLIND_TEST
...,...,...
35_9-7,train,BLIND_TEST
35_9-8,train,PUBLIC
36_7-3,train,TRAIN



Dataset SPLIT counts:
SPLIT
train         94
validation    24
Name: count, dtype: int64

Official manifest counts:
SPLIT
TRAIN         98
PUBLIC        10
BLIND_TEST    10
Name: count, dtype: int64


In [10]:
# ============================================================
# Cell 3 — Apply and verify OFFICIAL FORCE 2020 split
# ============================================================

# Remove stale temporary split
df = df.drop(columns=["SPLIT"], errors="ignore")

# Attach permanent official split
df = df.merge(
    split_manifest[["WELL", "SPLIT"]],
    on="WELL",
    how="left",
    validate="many_to_one"
)

# ------------------------------------------------------------
# Verify every well has exactly one official assignment
# ------------------------------------------------------------

well_split = df[["WELL", "SPLIT"]].drop_duplicates()

assert well_split["WELL"].nunique() == 118
assert well_split["SPLIT"].notna().all()

# Verify official WELL counts
well_counts = well_split["SPLIT"].value_counts()

assert well_counts["TRAIN"] == 98
assert well_counts["PUBLIC"] == 10
assert well_counts["BLIND_TEST"] == 10

print("✓ Official split applied successfully.")
print("\nWells per split:")
print(well_counts)

# ------------------------------------------------------------
# Verify sample counts
# ------------------------------------------------------------

sample_counts = df["SPLIT"].value_counts()

print("\nSamples per split:")
print(sample_counts)

# Expected official sample counts
assert sample_counts["TRAIN"] == 1_171_557
assert sample_counts["PUBLIC"] == 136_841
assert sample_counts["BLIND_TEST"] == 122_576

print("\n✓ Sample counts verified.")
print("✓ WELL counts verified.")
print("✓ Official FORCE 2020 split is ready.")

✓ Official split applied successfully.

Wells per split:
SPLIT
TRAIN         98
PUBLIC        10
BLIND_TEST    10
Name: count, dtype: int64

Samples per split:
SPLIT
TRAIN         1171557
PUBLIC         136841
BLIND_TEST     122576
Name: count, dtype: int64

✓ Sample counts verified.
✓ WELL counts verified.
✓ Official FORCE 2020 split is ready.


In [12]:
# ============================================================
# Diagnose final W5 feature definition
# ============================================================

print("Baseline features:", len(BASELINE_FEATURES))
print("Rolling features:", len(ROLLING_FEATURES))
print("Missing features:", len(MISSING_FEATURES))
print("Total:", len(BASELINE_FEATURES) + len(ROLLING_FEATURES) + len(MISSING_FEATURES))

print("\nRolling features:")
for i, col in enumerate(ROLLING_FEATURES, 1):
    print(f"{i:2d}. {col}")

print("\nMissing features:")
for i, col in enumerate(MISSING_FEATURES, 1):
    print(f"{i:2d}. {col}")

print("\nColumns containing ROLL:")
print([c for c in df.columns if "ROLL" in c.upper()])

Baseline features: 5
Rolling features: 30
Missing features: 5
Total: 40

Rolling features:
 1. GR_ROLLMEAN_5
 2. GR_ROLLSTD_5
 3. GR_ROLLMEAN_9
 4. GR_ROLLSTD_9
 5. GR_ROLLMEAN_21
 6. GR_ROLLSTD_21
 7. RDEP_LOG10_ROLLMEAN_5
 8. RDEP_LOG10_ROLLSTD_5
 9. RDEP_LOG10_ROLLMEAN_9
10. RDEP_LOG10_ROLLSTD_9
11. RDEP_LOG10_ROLLMEAN_21
12. RDEP_LOG10_ROLLSTD_21
13. RMED_LOG10_ROLLMEAN_5
14. RMED_LOG10_ROLLSTD_5
15. RMED_LOG10_ROLLMEAN_9
16. RMED_LOG10_ROLLSTD_9
17. RMED_LOG10_ROLLMEAN_21
18. RMED_LOG10_ROLLSTD_21
19. DTC_ROLLMEAN_5
20. DTC_ROLLSTD_5
21. DTC_ROLLMEAN_9
22. DTC_ROLLSTD_9
23. DTC_ROLLMEAN_21
24. DTC_ROLLSTD_21
25. RHOB_ROLLMEAN_5
26. RHOB_ROLLSTD_5
27. RHOB_ROLLMEAN_9
28. RHOB_ROLLSTD_9
29. RHOB_ROLLMEAN_21
30. RHOB_ROLLSTD_21

Missing features:
 1. GR_MISSING
 2. RDEP_LOG10_MISSING
 3. RMED_LOG10_MISSING
 4. DTC_MISSING
 5. RHOB_MISSING

Columns containing ROLL:
['GR_ROLLMEAN_5', 'GR_ROLLSTD_5', 'GR_ROLLMEAN_9', 'GR_ROLLSTD_9', 'GR_ROLLMEAN_21', 'GR_ROLLSTD_21', 'RDEP_LOG10_ROLLMEA

In [13]:
# ============================================================
# Cell 4 — Define the FROZEN W5 feature set
# ============================================================

TARGET = "lithology"

# ------------------------------------------------------------
# 1. Baseline features — 5
# ------------------------------------------------------------

BASELINE_FEATURES = [
    "GR",
    "RDEP_LOG10",
    "RMED_LOG10",
    "DTC",
    "RHOB",
]

# ------------------------------------------------------------
# 2. Rolling features
#    5 logs × 3 windows × 2 statistics = 30
# ------------------------------------------------------------

ROLLING_FEATURES = []

for feature in BASELINE_FEATURES:
    for window in [5, 9, 21]:
        ROLLING_FEATURES.append(f"{feature}_ROLLMEAN_{window}")
        ROLLING_FEATURES.append(f"{feature}_ROLLSTD_{window}")

# ------------------------------------------------------------
# 3. Resistivity separation — 1
# ------------------------------------------------------------

RESISTIVITY_FEATURE = [
    "RESISTIVITY_SEPARATION"
]

# ------------------------------------------------------------
# 4. Missingness indicators — 5
# ------------------------------------------------------------

MISSING_FEATURES = [
    "GR_MISSING",
    "RDEP_LOG10_MISSING",
    "RMED_LOG10_MISSING",
    "DTC_MISSING",
    "RHOB_MISSING",
]

# ------------------------------------------------------------
# Complete frozen W5 feature list
# ------------------------------------------------------------

ENGINEERED_FEATURES = (
    BASELINE_FEATURES
    + ROLLING_FEATURES
    + RESISTIVITY_FEATURE
    + MISSING_FEATURES
)

print("Feature counts:")
print("  Baseline:", len(BASELINE_FEATURES))
print("  Rolling:", len(ROLLING_FEATURES))
print("  Resistivity:", len(RESISTIVITY_FEATURE))
print("  Missing indicators:", len(MISSING_FEATURES))
print("  TOTAL:", len(ENGINEERED_FEATURES))

# ------------------------------------------------------------
# Verify every feature exists
# ------------------------------------------------------------

missing_columns = [
    col for col in ENGINEERED_FEATURES
    if col not in df.columns
]

print("\nMissing feature columns:", missing_columns)

assert len(ENGINEERED_FEATURES) == 41
assert len(missing_columns) == 0

print("\n✓ Frozen W5 feature definition verified.")

Feature counts:
  Baseline: 5
  Rolling: 30
  Resistivity: 1
  Missing indicators: 5
  TOTAL: 41

Missing feature columns: []

✓ Frozen W5 feature definition verified.


In [16]:
# ============================================================
# Cell 5 — Build final 98-well training matrix
# ============================================================

TARGET = "FORCE_2020_LITHOFACIES_LITHOLOGY"

train_mask = df["SPLIT"].eq("TRAIN")

# ------------------------------------------------------------
# Features
# ------------------------------------------------------------

X_train_final = df.loc[
    train_mask,
    ENGINEERED_FEATURES
].copy()

# ------------------------------------------------------------
# Target
# ------------------------------------------------------------

y_train_final = df.loc[
    train_mask,
    TARGET
].copy()

# Keep WELL separately for verification
train_wells = df.loc[
    train_mask,
    "WELL"
].copy()

# ------------------------------------------------------------
# Basic information
# ------------------------------------------------------------

print("X_train_final shape:", X_train_final.shape)
print("y_train_final shape:", y_train_final.shape)
print("Unique training wells:", train_wells.nunique())

print("\nMissing values before final imputation:")
print(
    X_train_final.isna()
    .sum()
    .sort_values(ascending=False)
)

print("\nTarget distribution:")
print(
    y_train_final
    .value_counts()
    .sort_index()
)

# ------------------------------------------------------------
# Safety checks
# ------------------------------------------------------------

assert X_train_final.shape == (1_171_557, 41)
assert y_train_final.shape[0] == 1_171_557
assert train_wells.nunique() == 98
assert y_train_final.notna().all()

print("\n✓ Final 98-well training matrix created successfully.")

X_train_final shape: (1171557, 41)
y_train_final shape: (1171557,)
Unique training wells: 98

Missing values before final imputation:
RHOB                      161482
RHOB_ROLLMEAN_5           161479
RHOB_ROLLSTD_5            161479
RHOB_ROLLMEAN_9           161109
RHOB_ROLLSTD_9            161109
RHOB_ROLLSTD_21           160152
RHOB_ROLLMEAN_21          160152
DTC_ROLLMEAN_5             81470
DTC                        81470
DTC_ROLLSTD_5              81470
DTC_ROLLSTD_9              81238
DTC_ROLLMEAN_9             81238
DTC_ROLLSTD_21             80564
DTC_ROLLMEAN_21            80564
RESISTIVITY_SEPARATION     39693
RMED_LOG10                 39013
RMED_LOG10_ROLLMEAN_5      38984
RMED_LOG10_ROLLSTD_5       38984
RMED_LOG10_ROLLMEAN_9      38294
RMED_LOG10_ROLLSTD_9       38294
RMED_LOG10_ROLLMEAN_21     37392
RMED_LOG10_ROLLSTD_21      37392
RDEP_LOG10                 11017
RDEP_LOG10_ROLLSTD_5       11015
RDEP_LOG10_ROLLMEAN_5      11015
RDEP_LOG10_ROLLMEAN_9      10952
RDEP_LOG

In [15]:
# ============================================================
# Diagnose target availability
# ============================================================

print("Columns containing 'lith':")
print([
    c for c in df.columns
    if "lith" in c.lower()
])

print("\nFirst 30 columns:")
print(df.columns[:30].tolist())

print("\nTotal columns:", len(df.columns))

Columns containing 'lith':
['FORCE_2020_LITHOFACIES_LITHOLOGY']

First 30 columns:
['GR', 'RDEP', 'RMED', 'DTC', 'RHOB', 'FORCE_2020_LITHOFACIES_LITHOLOGY', 'WELL', 'RDEP_LOG10', 'RMED_LOG10', 'GR_MISSING', 'RDEP_LOG10_MISSING', 'RMED_LOG10_MISSING', 'DTC_MISSING', 'RHOB_MISSING', 'GR_ROLLMEAN_5', 'GR_ROLLSTD_5', 'GR_ROLLMEAN_9', 'GR_ROLLSTD_9', 'GR_ROLLMEAN_21', 'GR_ROLLSTD_21', 'RDEP_LOG10_ROLLMEAN_5', 'RDEP_LOG10_ROLLSTD_5', 'RDEP_LOG10_ROLLMEAN_9', 'RDEP_LOG10_ROLLSTD_9', 'RDEP_LOG10_ROLLMEAN_21', 'RDEP_LOG10_ROLLSTD_21', 'RMED_LOG10_ROLLMEAN_5', 'RMED_LOG10_ROLLSTD_5', 'RMED_LOG10_ROLLMEAN_9', 'RMED_LOG10_ROLLSTD_9']

Total columns: 46


In [17]:
# ============================================================
# Cell 6 — Final training-only imputation
# ============================================================

# Continuous features only
CONTINUOUS_FEATURES = (
    BASELINE_FEATURES
    + ROLLING_FEATURES
    + RESISTIVITY_FEATURE
)

# Calculate medians ONLY from the 98 official TRAIN wells
final_train_medians = (
    X_train_final[CONTINUOUS_FEATURES]
    .median()
)

print("Final training medians:")
display(final_train_medians.to_frame("median"))

# ------------------------------------------------------------
# Apply imputation
# ------------------------------------------------------------

X_train_final[CONTINUOUS_FEATURES] = (
    X_train_final[CONTINUOUS_FEATURES]
    .fillna(final_train_medians)
)

# Missing indicators are already 0/1 and contain no NaNs
assert X_train_final[MISSING_FEATURES].isna().sum().sum() == 0

# ------------------------------------------------------------
# Final checks
# ------------------------------------------------------------

remaining_nan = X_train_final.isna().sum().sum()
remaining_inf = np.isinf(X_train_final.to_numpy()).sum()

print("\nRemaining NaN values:", remaining_nan)
print("Remaining infinite values:", remaining_inf)

assert remaining_nan == 0
assert remaining_inf == 0

print("\n✓ Final TRAIN matrix completely imputed.")
print("✓ Imputation parameters derived only from the 98 TRAIN wells.")

Final training medians:


,median
GR,68.367630
RDEP_LOG10,0.158287
RMED_LOG10,0.159701
DTC,109.560997
RHOB,2.321426
GR_ROLLMEAN_5,68.500006
GR_ROLLSTD_5,2.025031
GR_ROLLMEAN_9,68.583333
GR_ROLLSTD_9,2.752253
GR_ROLLMEAN_21,68.684045



Remaining NaN values: 0
Remaining infinite values: 0

✓ Final TRAIN matrix completely imputed.
✓ Imputation parameters derived only from the 98 TRAIN wells.


In [18]:
# ============================================================
# Cell 7 — Frozen W5 √-balanced class weights
# ============================================================

# Number of classes
n_classes = y_train_final.nunique()

# Class counts from the complete 98-well TRAIN set
class_counts = y_train_final.value_counts().sort_index()

# Standard sklearn "balanced" class weight:
# n_samples / (n_classes * n_samples_for_class)
balanced_weights = (
    len(y_train_final)
    / (n_classes * class_counts)
)

# W5 = square-root of balanced weights
sqrt_balanced_weights = np.sqrt(balanced_weights)

# Convert to dictionary using the ORIGINAL lithology codes
w5_class_weights = sqrt_balanced_weights.to_dict()

print("W5 √-balanced class weights:\n")

for lithology, weight in w5_class_weights.items():
    print(
        f"{int(lithology):5d} : "
        f"{weight:.6f} "
        f"(n = {class_counts[lithology]:,})"
    )

print("\nWeight statistics:")
print("Minimum:", sqrt_balanced_weights.min())
print("Maximum:", sqrt_balanced_weights.max())
print("Mean:", sqrt_balanced_weights.mean())
print("Median:", sqrt_balanced_weights.median())

assert len(w5_class_weights) == 12

print("\n✓ Frozen W5 class weights calculated.")

W5 √-balanced class weights:

30000 : 0.759151 (n = 169,405)
65000 : 0.367932 (n = 721,185)
65030 : 0.805140 (n = 150,605)
70000 : 1.316618 (n = 56,320)
70032 : 3.047388 (n = 10,513)
74000 : 7.605099 (n = 1,688)
80000 : 1.711332 (n = 33,336)
86000 : 9.485849 (n = 1,085)
88000 : 3.447575 (n = 8,214)
90000 : 5.055445 (n = 3,820)
93000 : 26.313677 (n = 141)
99000 : 2.530623 (n = 15,245)

Weight statistics:
Minimum: 0.36793215657250794
Maximum: 26.313676566912598
Mean: 5.20381905787283
Median: 2.7890052885114445

✓ Frozen W5 class weights calculated.


In [19]:
# ============================================================
# Cell 8 — Train FINAL W5 Extra Trees model
# ============================================================

# Convert lithology-code weights into per-sample weights
sample_weights_final = y_train_final.map(
    w5_class_weights
).to_numpy()

assert len(sample_weights_final) == len(y_train_final)
assert np.isfinite(sample_weights_final).all()

print("Sample-weight statistics:")
print("Minimum:", sample_weights_final.min())
print("Maximum:", sample_weights_final.max())
print("Mean:", sample_weights_final.mean())

# ------------------------------------------------------------
# Frozen W5 model
# ------------------------------------------------------------

final_w5_model = ExtraTreesClassifier(
    n_estimators=100,
    max_depth=20,
    min_samples_leaf=5,
    class_weight=None,       # weights supplied explicitly
    n_jobs=-1,
    random_state=42
)

print("\nTraining final W5 model...")
start_time = time.time()

final_w5_model.fit(
    X_train_final,
    y_train_final,
    sample_weight=sample_weights_final
)

train_time = time.time() - start_time

print(f"\n✓ Final W5 model trained.")
print(f"Training time: {train_time:.2f} seconds")

print("\nModel:")
print(final_w5_model)


Sample-weight statistics:
Minimum: 0.36793215657250794
Maximum: 26.313676566912598
Mean: 0.675593734179306

Training final W5 model...

✓ Final W5 model trained.
Training time: 36.40 seconds

Model:
ExtraTreesClassifier(max_depth=20, min_samples_leaf=5, n_jobs=-1,
                     random_state=42)


In [20]:
# ============================================================
# Cell 9 — Final W5 evaluation on PUBLIC
# ============================================================

public_mask = df["SPLIT"].eq("PUBLIC")

X_public_final = df.loc[
    public_mask,
    ENGINEERED_FEATURES
].copy()

y_public_final = df.loc[
    public_mask,
    TARGET
].copy()

public_wells = df.loc[
    public_mask,
    "WELL"
].nunique()

# ------------------------------------------------------------
# Apply FINAL TRAINING medians
# ------------------------------------------------------------

X_public_final[CONTINUOUS_FEATURES] = (
    X_public_final[CONTINUOUS_FEATURES]
    .fillna(final_train_medians)
)

# ------------------------------------------------------------
# Safety checks
# ------------------------------------------------------------

assert X_public_final.shape == (136_841, 41)
assert y_public_final.notna().all()
assert X_public_final.isna().sum().sum() == 0
assert np.isfinite(X_public_final.to_numpy()).all()

print("PUBLIC samples:", len(X_public_final))
print("PUBLIC wells:", public_wells)

# ------------------------------------------------------------
# Prediction
# ------------------------------------------------------------

start_time = time.time()

y_public_pred = final_w5_model.predict(X_public_final)

public_prediction_time = time.time() - start_time

# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------

public_accuracy = accuracy_score(
    y_public_final,
    y_public_pred
)

public_balanced_accuracy = balanced_accuracy_score(
    y_public_final,
    y_public_pred
)

public_macro_f1 = f1_score(
    y_public_final,
    y_public_pred,
    average="macro"
)

public_weighted_f1 = f1_score(
    y_public_final,
    y_public_pred,
    average="weighted"
)

print("\n" + "=" * 55)
print("FINAL W5 — PUBLIC RESULTS")
print("=" * 55)

print(f"Accuracy:          {public_accuracy:.4f}")
print(f"Balanced Accuracy: {public_balanced_accuracy:.4f}")
print(f"Macro F1:          {public_macro_f1:.4f}")
print(f"Weighted F1:       {public_weighted_f1:.4f}")
print(f"Prediction time:   {public_prediction_time:.2f} s")

PUBLIC samples: 136841
PUBLIC wells: 10

FINAL W5 — PUBLIC RESULTS
Accuracy:          0.7476
Balanced Accuracy: 0.3971
Macro F1:          0.3447
Weighted F1:       0.7390
Prediction time:   1.76 s


/Users/akashray/Lithology_Predictor/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


| Metric | Final W5 PUBLIC |
|---|---:|
| Accuracy | **0.7476** |
| Balanced Accuracy | **0.3971** |
| Macro F1 | **0.3447** |
| Weighted F1 | **0.7390** |
| Prediction time | **1.76 s** |

In [21]:
# ============================================================
# Cell 10 — Final PUBLIC classification report
# ============================================================

labels = sorted(y_train_final.unique())

print("=" * 70)
print("FINAL W5 — PUBLIC CLASSIFICATION REPORT")
print("=" * 70)

public_report = classification_report(
    y_public_final,
    y_public_pred,
    labels=labels,
    target_names=[str(int(x)) for x in labels],
    zero_division=0
)

print(public_report)

# ------------------------------------------------------------
# Confusion matrix
# ------------------------------------------------------------

cm_public = confusion_matrix(
    y_public_final,
    y_public_pred,
    labels=labels
)

print("\nConfusion matrix:")
print(cm_public)

FINAL W5 — PUBLIC CLASSIFICATION REPORT
              precision    recall  f1-score   support

       30000       0.79      0.82      0.81     24048
       65000       0.84      0.87      0.86     84030
       65030       0.40      0.28      0.33     17558
       70000       0.47      0.52      0.50      4798
       70032       0.64      0.01      0.02       625
       74000       0.00      0.00      0.00       416
       80000       0.15      0.24      0.19      3306
       86000       1.00      0.01      0.02       125
       88000       0.00      0.00      0.00         0
       90000       0.52      0.82      0.64       690
       93000       0.00      0.00      0.00         0
       99000       0.51      0.40      0.45      1245

    accuracy                           0.75    136841
   macro avg       0.44      0.33      0.32    136841
weighted avg       0.74      0.75      0.74    136841


Confusion matrix:
[[19742  1643  1684   476     0     0   209     0     0   144     0   150]

In [22]:
# ============================================================
# Cell 11 — Audit final PUBLIC metrics
# ============================================================

print("Unique TRUE classes:")
print(np.sort(np.unique(y_public_final)))

print("\nUnique PREDICTED classes:")
print(np.sort(np.unique(y_public_pred)))

# Explicitly use ALL 12 training classes
final_labels = np.sort(y_train_final.unique())

audit_accuracy = accuracy_score(
    y_public_final,
    y_public_pred
)

audit_balanced_accuracy = balanced_accuracy_score(
    y_public_final,
    y_public_pred
)

audit_macro_f1 = f1_score(
    y_public_final,
    y_public_pred,
    labels=final_labels,
    average="macro",
    zero_division=0
)

audit_weighted_f1 = f1_score(
    y_public_final,
    y_public_pred,
    labels=final_labels,
    average="weighted",
    zero_division=0
)

print("\n" + "=" * 60)
print("AUDITED FINAL PUBLIC METRICS")
print("=" * 60)

print(f"Accuracy:          {audit_accuracy:.6f}")
print(f"Balanced Accuracy: {audit_balanced_accuracy:.6f}")
print(f"Macro F1:          {audit_macro_f1:.6f}")
print(f"Weighted F1:       {audit_weighted_f1:.6f}")

print("\nClassification-report macro F1:")
print(
    f1_score(
        y_public_final,
        y_public_pred,
        labels=final_labels,
        average="macro",
        zero_division=0
    )
)

Unique TRUE classes:
[30000. 65000. 65030. 70000. 70032. 74000. 80000. 86000. 90000. 99000.]

Unique PREDICTED classes:
[30000. 65000. 65030. 70000. 70032. 80000. 86000. 88000. 90000. 99000.]


/Users/akashray/Lithology_Predictor/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")



AUDITED FINAL PUBLIC METRICS
Accuracy:          0.747612
Balanced Accuracy: 0.397069
Macro F1:          0.316008
Weighted F1:       0.738985

Classification-report macro F1:
0.3160082322465744


| Metric | Final W5 |
|---|---:|
| Accuracy | **0.747612** |
| Balanced Accuracy | **0.397069** |
| **Macro F1** | **0.316008** |
| Weighted F1 | **0.738985** |

In [23]:
# ============================================================
# Cell 12 — Save FINAL W5 model and development artifacts
# ============================================================

import joblib

# ------------------------------------------------------------
# Create final result directories
# ------------------------------------------------------------

MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. Save trained model
# ------------------------------------------------------------

model_path = MODEL_DIR / "lithology_w5_final.joblib"

joblib.dump(
    final_w5_model,
    model_path
)

# ------------------------------------------------------------
# 2. Save final preprocessing parameters
# ------------------------------------------------------------

preprocessing_final = {
    "continuous_features": CONTINUOUS_FEATURES,
    "missing_features": MISSING_FEATURES,
    "training_medians": {
        str(k): float(v)
        for k, v in final_train_medians.items()
    }
}

preprocessing_path = MODEL_DIR / "preprocessing_params_final.json"

with open(preprocessing_path, "w") as f:
    json.dump(preprocessing_final, f, indent=2)

# ------------------------------------------------------------
# 3. Save feature definition
# ------------------------------------------------------------

feature_metadata = {
    "n_features": len(ENGINEERED_FEATURES),
    "features": ENGINEERED_FEATURES,
    "target": TARGET,
    "model_name": "W5_ExtraTrees_sqrt_balanced",
    "random_state": 42
}

feature_path = MODEL_DIR / "w5_feature_metadata.json"

with open(feature_path, "w") as f:
    json.dump(feature_metadata, f, indent=2)

# ------------------------------------------------------------
# 4. Save class weights
# ------------------------------------------------------------

class_weight_path = MODEL_DIR / "w5_class_weights.json"

with open(class_weight_path, "w") as f:
    json.dump(
        {
            str(int(k)): float(v)
            for k, v in w5_class_weights.items()
        },
        f,
        indent=2
    )

# ------------------------------------------------------------
# 5. Save final PUBLIC metrics
# ------------------------------------------------------------

final_public_metrics = {
    "model": "W5_ExtraTrees_sqrt_balanced",
    "n_training_wells": 98,
    "n_training_samples": int(len(X_train_final)),
    "n_features": len(ENGINEERED_FEATURES),
    "public_wells": 10,
    "public_samples": int(len(X_public_final)),
    "accuracy": float(audit_accuracy),
    "balanced_accuracy": float(audit_balanced_accuracy),
    "macro_f1": float(audit_macro_f1),
    "weighted_f1": float(audit_weighted_f1),
    "training_time_seconds": float(train_time),
    "prediction_time_seconds": float(public_prediction_time)
}

metrics_path = RESULTS_DIR / "final_public_metrics.json"

with open(metrics_path, "w") as f:
    json.dump(final_public_metrics, f, indent=2)

# ------------------------------------------------------------
# 6. Save classification report
# ------------------------------------------------------------

report_dict = classification_report(
    y_public_final,
    y_public_pred,
    labels=final_labels,
    target_names=[str(int(x)) for x in final_labels],
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(report_dict).T

report_path = RESULTS_DIR / "final_public_classification_report.csv"

report_df.to_csv(report_path)

# ------------------------------------------------------------
# 7. Save confusion matrix
# ------------------------------------------------------------

cm_df = pd.DataFrame(
    cm_public,
    index=[str(int(x)) for x in final_labels],
    columns=[str(int(x)) for x in final_labels]
)

cm_path = RESULTS_DIR / "final_public_confusion_matrix.csv"

cm_df.to_csv(cm_path)

# ------------------------------------------------------------
# 8. Save PUBLIC predictions
# ------------------------------------------------------------

public_predictions = df.loc[
    public_mask,
    ["WELL"]
].copy()

public_predictions["TRUE_LITHOLOGY"] = y_public_final.to_numpy()
public_predictions["PREDICTED_LITHOLOGY"] = y_public_pred

predictions_path = RESULTS_DIR / "final_public_predictions.parquet"

public_predictions.to_parquet(
    predictions_path,
    index=False
)

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("✓ Final W5 model saved:")
print(model_path)

print("\n✓ Preprocessing parameters saved:")
print(preprocessing_path)

print("\n✓ Feature metadata saved:")
print(feature_path)

print("\n✓ Class weights saved:")
print(class_weight_path)

print("\n✓ PUBLIC metrics saved:")
print(metrics_path)

print("\n✓ Classification report saved:")
print(report_path)

print("\n✓ Confusion matrix saved:")
print(cm_path)

print("\n✓ PUBLIC predictions saved:")
print(predictions_path)

print("\n" + "=" * 60)
print("FINAL DEVELOPMENT ARTIFACTS SAVED")
print("=" * 60)

✓ Final W5 model saved:
../models/lithology_w5_final.joblib

✓ Preprocessing parameters saved:
../models/preprocessing_params_final.json

✓ Feature metadata saved:
../models/w5_feature_metadata.json

✓ Class weights saved:
../models/w5_class_weights.json

✓ PUBLIC metrics saved:
../results/final_public_metrics.json

✓ Classification report saved:
../results/final_public_classification_report.csv

✓ Confusion matrix saved:
../results/final_public_confusion_matrix.csv

✓ PUBLIC predictions saved:
../results/final_public_predictions.parquet

FINAL DEVELOPMENT ARTIFACTS SAVED
